# Text Classification & Sentiment Analysis — the demos, executed

Every number, table, and figure on the [concept page](../10-Text-Classification-and-Sentiment-Analysis.md) is produced by the seeded functions in `text_classification.py`, which this notebook imports. Nothing here downloads a dataset: the sentiment corpus is generated deterministically from polarity-word pools, so the whole thing runs on CPU in a couple of seconds and reproduces bit-for-bit.

**One idea per cell, and every cell asserts its qualitative point _before_ it prints.** The demos, in order:
1. Multinomial NB by hand, proven equal to scikit-learn.
2. The classical ladder: NB vs LogReg vs SVM (LogReg wins).
3. The confusion matrix and precision/recall/F1 from its counts.
4. The threshold sweep — precision vs recall trade-off.
5. Class imbalance: accuracy lies, F1 / PR-AUC don't.
6. PR vs ROC under imbalance.
7. Train/test leakage from fitting preprocessing on test.

In [1]:
# Imports come from the single source-of-truth module, so the notebook and the page agree.
from text_classification import (
    device_line,
    logreg_tiny_pos_proba,
    nb_by_hand,
    nb_sklearn_joint_log_lik,
    precision_recall_f1_from_counts,
    run_imbalance_demo,
    run_ladder_comparison,
    run_leakage_demo,
    stable_token_hash,
    threshold_sweep,
)

print(device_line())

python 3.12.13 · sklearn 1.9.0 · numpy 2.4.6 · platform Darwin (CPU)


## 1. Multinomial Naive Bayes, by hand — and proven against scikit-learn

Classify the document `'great fun'` against the tiny 4-document corpus by computing the log-posterior `log P(c) + Σ count·log P(w|c)` with add-1 smoothing, then read scikit-learn's joint log-likelihood and check they match to 4 decimals.

In [2]:
bh = nb_by_hand()
neg_sk, pos_sk = nb_sklearn_joint_log_lik()

# Assert the qualitative point FIRST: by-hand == sklearn, and the prediction is positive.
assert bh.predicted == 1, "'great fun' should be classified positive"
assert abs(bh.logp_pos - pos_sk) < 1e-4 and abs(bh.logp_neg - neg_sk) < 1e-4

print(f"by hand : logp_pos={bh.logp_pos:.4f}  logp_neg={bh.logp_neg:.4f}  -> pos")
print(f"sklearn : jll_pos ={pos_sk:.4f}  jll_neg ={neg_sk:.4f}  -> pos")
print(f"match: True   gap in log-space = {abs(bh.logp_pos - bh.logp_neg):.2f}")

by hand : logp_pos=-3.3604  logp_neg=-5.6630  -> pos
sklearn : jll_pos =-3.3604  jll_neg =-5.6630  -> pos
match: True   gap in log-space = 2.30


### 1b. The same document under logistic regression — a calmer probability

On the *same* tiny corpus, fit TF-IDF + LogisticRegression and read `P(pos | 'great fun')`. It agrees with NB (positive) but its probability is **calmer** (~0.62 vs NB's effective ~0.9) — NB's false independence assumption double-counts correlated evidence and pushes probabilities toward 0/1.

In [3]:
lr_pos = logreg_tiny_pos_proba()

# Assert the qualitative point FIRST: LogReg also calls it positive, but with a calmer probability.
assert lr_pos > 0.5, "LogReg should call 'great fun' positive"
assert lr_pos < 0.8, "LogReg's probability should be calmer than NB's effective ~0.9"

print(f"TF-IDF + LogReg:  P(pos | 'great fun') = {lr_pos:.3f}  -> pos")
print("(calmer than NB's effective ~0.9 -- LogReg is better calibrated here)")

TF-IDF + LogReg:  P(pos | 'great fun') = 0.620  -> pos
(calmer than NB's effective ~0.9 -- LogReg is better calibrated here)


## 2. The classical ladder: NB vs LogReg vs SVM

Train all three on the balanced synthetic set (720 train / 480 test). The corpus has **burst-correlated** word clusters that violate NB's independence assumption, so LogReg — which down-weights the redundant features — edges out NB.

In [4]:
results, lr, x_test, y_test = run_ladder_comparison()
by_name = {r.name: r for r in results}

# Assert the qualitative point FIRST: LogReg beats NB, both are strong baselines.
assert by_name["TF-IDF + LogReg"].accuracy > by_name["Multinomial NB"].accuracy
assert all(r.accuracy > 0.85 for r in results)

for r in results:
    print(f"{r.name:<18} acc={r.accuracy:.3f}  macroF1={r.macro_f1:.3f}  "
          f"PR-AUC={r.pr_auc:.3f}  ROC-AUC={r.roc_auc:.3f}")

Multinomial NB     acc=0.906  macroF1=0.906  PR-AUC=0.974  ROC-AUC=0.971
TF-IDF + LogReg    acc=0.919  macroF1=0.919  PR-AUC=0.977  ROC-AUC=0.972
Linear SVM         acc=0.890  macroF1=0.890  PR-AUC=0.968  ROC-AUC=0.964


## 3. The confusion matrix, and precision/recall/F1 from its counts

The confusion matrix is the source of truth; every scalar metric is a summary of its four cells. Derive precision, recall, and F1 for the positive class straight from TP/FP/FN.

In [5]:
cm = by_name["TF-IDF + LogReg"].cm  # rows=actual [neg,pos], cols=pred [neg,pos]
tn, fp, fn, tp = int(cm[0, 0]), int(cm[0, 1]), int(cm[1, 0]), int(cm[1, 1])
precision, recall, f1 = precision_recall_f1_from_counts(tp, fp, fn)

# Assert the qualitative point FIRST: a balanced set gives near-symmetric errors, high F1.
assert abs(fp - fn) < 0.5 * max(fp, fn), "errors should be roughly symmetric on a balanced set"
assert f1 > 0.9

print(f"confusion matrix [rows=actual, cols=pred]:\n{cm}")
print(f"TP={tp} FP={fp} FN={fn} TN={tn}")
print(f"precision={precision:.3f}  recall={recall:.3f}  F1={f1:.3f}")

confusion matrix [rows=actual, cols=pred]:
[[217  17]
 [ 22 224]]
TP=224 FP=17 FN=22 TN=217
precision=0.929  recall=0.911  F1=0.920


## 4. The threshold sweep — precision vs recall trade-off

A model outputs a probability; the default 0.5 is just one column of this table. As the threshold rises, precision rises and recall falls; F1 peaks in between.

In [6]:
import numpy as np

scores = lr.scores(x_test)
sweep = threshold_sweep(scores, y_test)
best_idx = int(np.argmax(sweep["f1"]))

# Assert the qualitative point FIRST: precision rises and recall falls as threshold increases.
lo = sweep["thresholds"] < 0.3
hi = sweep["thresholds"] > 0.7
assert sweep["precision"][hi].mean() > sweep["precision"][lo].mean()
assert sweep["recall"][lo].mean() > sweep["recall"][hi].mean()

for thr in (0.2, 0.5, 0.8):
    i = int(np.argmin(np.abs(sweep["thresholds"] - thr)))
    print(f"thr={sweep['thresholds'][i]:.2f}  P={sweep['precision'][i]:.3f}  "
          f"R={sweep['recall'][i]:.3f}  F1={sweep['f1'][i]:.3f}")
print(f"best-F1 threshold = {sweep['thresholds'][best_idx]:.2f} "
      f"(F1={sweep['f1'][best_idx]:.3f})")

thr=0.20  P=0.750  R=0.988  F1=0.853
thr=0.50  P=0.929  R=0.911  F1=0.920
thr=0.80  P=1.000  R=0.606  F1=0.754
best-F1 threshold = 0.48 (F1=0.923)


## 5. Class imbalance: accuracy lies, F1 / PR-AUC don't

On an 8%-positive set, a do-nothing 'always negative' classifier gets high accuracy yet is useless. Macro-F1 and PR-AUC expose it; a real class-weighted model wins on those.

In [7]:
imb = run_imbalance_demo()

# Assert the qualitative point FIRST: accuracy ranks the useless model high; F1/PR-AUC don't.
assert imb["majority_acc"] > 0.85, "do-nothing model looks deceptively accurate"
assert imb["model_f1"] > imb["majority_f1"]
assert imb["model_pr_auc"] > imb["majority_pr_auc"]

print(f"positive rate in test: {imb['pos_rate']:.3f}\n")
print(f"{'metric':<10}{'always-neg':>12}{'LogReg(bal)':>14}")
print(f"{'accuracy':<10}{imb['majority_acc']:>12.3f}{imb['model_acc']:>14.3f}")
print(f"{'macro-F1':<10}{imb['majority_f1']:>12.3f}{imb['model_f1']:>14.3f}")
print(f"{'PR-AUC':<10}{imb['majority_pr_auc']:>12.3f}{imb['model_pr_auc']:>14.3f}")
print("\n-> accuracy alone would pick the useless model; macro-F1 and PR-AUC do not.")

positive rate in test: 0.081

metric      always-neg   LogReg(bal)
accuracy         0.919         0.948
macro-F1         0.479         0.835
PR-AUC           0.081         0.764

-> accuracy alone would pick the useless model; macro-F1 and PR-AUC do not.


## 6. PR vs ROC under imbalance

Feed the _same_ scores to a ROC curve and a PR curve under heavy imbalance: ROC-AUC stays flattering (its FPR barely moves because the TN count is huge), while PR-AUC drops to the honest difficulty.

In [8]:
# ROC-AUC is much higher than PR-AUC for the same scores on the rare-positive set.
assert imb["model_roc_auc"] > imb["model_pr_auc"], \
    "under imbalance ROC flatters relative to PR"

print(f"same scores, {imb['pos_rate']:.0%}-positive test set:")
print(f"  ROC-AUC = {imb['model_roc_auc']:.3f}   (flattering: FPR insensitive to FP under big TN)")
print(f"  PR-AUC  = {imb['model_pr_auc']:.3f}   (honest: precision has no TN term)")
print("  rule: when the positive class is rare, trust the PR curve.")

same scores, 8%-positive test set:
  ROC-AUC = 0.968   (flattering: FPR insensitive to FP under big TN)
  PR-AUC  = 0.764   (honest: precision has no TN term)
  rule: when the positive class is rare, trust the PR curve.


## 7. Train/test leakage — preprocessing fit on test inflates the score

The same family as fitting a vectorizer on test: here a feature-selection step is fit on **train+test** labels. On a corpus of **pure noise** (the label is independent of the text) the honest answer is chance (~0.50); the leaky pipeline 'discovers' features that match the test labels by accident and reports a confidently-wrong accuracy. That whole gap is fabricated by the leak — it would vanish in production.

In [9]:
leak = run_leakage_demo()

# Assert the qualitative point FIRST: honest stays at chance (~0.50); the leak inflates well above it.
assert abs(leak["honest_acc"] - 0.5) < 0.06, "honest pipeline on pure noise should be near chance"
assert leak["leaky_acc"] > leak["honest_acc"] + 0.1, "leakage should noticeably inflate the score"

print(f"honest (select on train labels only) : acc={leak['honest_acc']:.3f}   (~chance, correct)")
print(f"leaky  (select on train+test labels) : acc={leak['leaky_acc']:.3f}   (fabricated)")
print(f"inflation from leakage               : {leak['gap']:+.3f}")
print("\nAlways fit every preprocessing step (vectorizer, selector, scaler) on TRAIN only.")

honest (select on train labels only) : acc=0.500   (~chance, correct)
leaky  (select on train+test labels) : acc=0.678   (fabricated)
inflation from leakage               : +0.178

Always fit every preprocessing step (vectorizer, selector, scaler) on TRAIN only.


## Appendix: stable hashing (never Python's salted `hash()`)

Any hashing-trick vectorizer must use a content-addressed hash, or it buckets words differently on every process. `stable_token_hash` is md5-based, so it is reproducible across runs and machines.

In [10]:
buckets = 1000
# Assert the qualitative point FIRST: the same token maps to the same bucket every time.
assert stable_token_hash("brilliant", buckets) == stable_token_hash("brilliant", buckets)
for token in ("brilliant", "terrible", "movie"):
    print(f"{token:<12} -> bucket {stable_token_hash(token, buckets)}")

brilliant    -> bucket 834
terrible     -> bucket 546
movie        -> bucket 455
